# 02 — Discrimination & Generalization

- **Stimulus Control**: discrimination index acquisition across sessions
- **Generalization**: gradient across theme similarity levels

Run `00_data_pipeline` first.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

ROOT     = Path('..').resolve()
PROC_DIR = ROOT / 'data' / 'processed'
FIG_DIR  = ROOT / 'reports' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11,
                     'axes.spines.top': False, 'axes.spines.right': False})

def load(name):
    p = PROC_DIR / f'{name}.parquet'
    return pd.read_parquet(p) if p.exists() else pd.DataFrame()


## 1. Stimulus Control — Discrimination Index

$$DI = \frac{\text{hits in SD}}{\text{blocks in SD}} - \frac{\text{hits in S}\Delta}{\text{blocks in S}\Delta}$$

DI = 1.0: perfect discrimination. DI = 0.0: no discrimination.


In [ ]:
sc = load('stimulus_control')

if not sc.empty:
    # Tag zone type from event codes
    # SD_PLUS / SD_MINUS mark zone entries
    # RESP_SD_PLUS / RESP_SD_MINUS mark hits within zones

    # Count hits and available blocks per zone type per session
    sc = sc.sort_values(['participant_id', 'session_id', 'elapsed_s'])
    sc['session_num'] = sc.groupby('participant_id')['session_id'] \
                          .transform(lambda x: pd.factorize(x)[0] + 1)

    rows = []
    for (pid, snum, sid), grp in sc.groupby(
            ['participant_id', 'session_num', 'session_id']):
        sd_zones   = grp[grp['event_code'] == 'SD_PLUS'].shape[0]
        sdel_zones = grp[grp['event_code'] == 'SD_MINUS'].shape[0]
        sd_hits    = grp[grp['event_code'] == 'RESP_SD_PLUS'].shape[0]
        sdel_hits  = grp[grp['event_code'] == 'RESP_SD_MINUS'].shape[0]

        # Blocks per zone = 5 per spec
        blocks_per_zone = 5
        sd_pct   = sd_hits   / max(sd_zones   * blocks_per_zone, 1)
        sdel_pct = sdel_hits / max(sdel_zones * blocks_per_zone, 1)
        di       = sd_pct - sdel_pct

        rows.append({'participant_id': pid, 'session_num': snum,
                     'sd_hit_rate': sd_pct, 'sdelta_hit_rate': sdel_pct,
                     'DI': di})
    di_df = pd.DataFrame(rows)

    display(di_df.round(3))

    # Plot DI acquisition
    fig, ax = plt.subplots(figsize=(7, 4))
    for pid, pgrp in di_df.groupby('participant_id'):
        pgrp = pgrp.sort_values('session_num')
        ax.plot(pgrp['session_num'], pgrp['DI'], marker='o', label=pid)
    ax.axhline(1.0, ls='--', color='green', alpha=0.4, label='Perfect (1.0)')
    ax.axhline(0.0, ls='--', color='gray',  alpha=0.4, label='Chance (0.0)')
    ax.set_xlabel('Session')
    ax.set_ylabel('Discrimination Index')
    ax.set_ylim(-0.1, 1.1)
    ax.set_title('Stimulus Control — DI Acquisition Across Sessions')
    ax.legend()
    fig.savefig(FIG_DIR / 'stimulus_control_DI.png', bbox_inches='tight')
    plt.show()


## 2. Stimulus Control — SD vs S∆ hit rate per session


In [ ]:
if 'di_df' in dir() and not di_df.empty:
    fig, ax = plt.subplots(figsize=(7, 4))
    sessions = sorted(di_df['session_num'].unique())
    x = np.arange(len(sessions))
    w = 0.35

    mean_sd   = [di_df[di_df['session_num'] == s]['sd_hit_rate'].mean()   for s in sessions]
    mean_sdel = [di_df[di_df['session_num'] == s]['sdelta_hit_rate'].mean() for s in sessions]

    ax.bar(x - w/2, mean_sd,   w, label='SD (night)',    color='#457b9d', alpha=0.85)
    ax.bar(x + w/2, mean_sdel, w, label='S∆ (sunset)',   color='#f4a261', alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels([f'Session {s}' for s in sessions])
    ax.set_ylabel('Hit rate (proportion of blocks hit)')
    ax.set_title('SD vs S∆ Hit Rate by Session')
    ax.legend()
    fig.savefig(FIG_DIR / 'stimulus_control_hit_rates.png', bbox_inches='tight')
    plt.show()


## 3. Generalization — gradient across themes

Expected gradient: Underground > Water > Castle > Airship

Similarity values from spec: `{Underground: 0.75, Water: 0.50, Castle: 0.25, Airship: 0.10}`


In [ ]:
gen = load('generalization')

SIMILARITY = {
    'Training':    1.00,
    'Underground': 0.75,
    'Water':       0.50,
    'Castle':      0.25,
    'Airship':     0.10,
}

if not gen.empty:
    # GEN_STIM events have the theme in the Note field
    gen_probes = gen[gen['event_code'] == 'GEN_STIM'].copy()
    gen_probes['theme'] = gen_probes['note'].str.strip().str.title()

    # For each probe segment, count hits (RESPONSE) that follow a GEN_STIM
    # until the next TRAIN_STIM or GEN_STIM
    gen = gen.sort_values(['session_id', 'elapsed_s'])
    gen['session_num'] = gen.groupby('participant_id')['session_id'] \
                           .transform(lambda x: pd.factorize(x)[0] + 1)

    gradient_rows = []
    blocks_per_segment = 5

    for (pid, snum, sid), grp in gen.groupby(
            ['participant_id', 'session_num', 'session_id']):
        grp = grp.sort_values('elapsed_s').reset_index(drop=True)
        current_theme = None
        segment_hits  = 0

        for _, row in grp.iterrows():
            if row['event_code'] in ('GEN_STIM', 'TRAIN_STIM'):
                if current_theme is not None and current_theme != 'Training':
                    gradient_rows.append({
                        'participant_id': pid, 'session_num': snum,
                        'theme': current_theme,
                        'hit_rate': segment_hits / blocks_per_segment,
                        'similarity': SIMILARITY.get(current_theme, np.nan)
                    })
                current_theme = row['note'].strip().title() if row['event_code'] == 'GEN_STIM' \
                    else 'Training'
                segment_hits = 0
            elif row['event_code'] == 'RESPONSE' and current_theme not in (None, 'Training'):
                segment_hits += 1

    if gradient_rows:
        grad_df = pd.DataFrame(gradient_rows)
        display(grad_df.groupby('theme').agg(
            hit_rate_mean=('hit_rate','mean'),
            hit_rate_sem=('hit_rate', lambda x: x.sem()),
            similarity=('similarity','first')
        ).sort_values('similarity', ascending=False).round(3))

        # Gradient plot
        summary = grad_df.groupby('theme').agg(
            mean=('hit_rate','mean'), sem=('hit_rate', lambda x: x.sem()),
            sim=('similarity','first')).sort_values('sim', ascending=False)

        fig, ax = plt.subplots(figsize=(7, 4))
        ax.errorbar(summary['sim'], summary['mean'], yerr=summary['sem'],
                    fmt='o-', color='#2a9d8f', capsize=4, linewidth=2)
        for theme, row in summary.iterrows():
            ax.annotate(theme, (row['sim'], row['mean']),
                        textcoords='offset points', xytext=(4, 4), fontsize=9)
        ax.set_xlabel('Stimulus similarity to training')
        ax.set_ylabel('Hit rate (proportion of blocks hit)')
        ax.set_xlim(-0.05, 1.05)
        ax.set_ylim(-0.05, 1.05)
        ax.set_title('Generalization Gradient')
        fig.savefig(FIG_DIR / 'generalization_gradient.png', bbox_inches='tight')
        plt.show()

        # Pearson correlation: hit rate ~ similarity
        r, p = pearsonr(grad_df['similarity'], grad_df['hit_rate'])
        print(f'\nCorrelation (hit rate ~ similarity): r = {r:.3f}, p = {p:.4f}')
    else:
        print('No probe segment data found. Check that GEN_STIM events have theme in Note field.')
